# Deploy Your Agent with FastAPI

**What you will build:** the same agent you've been running in cells, exposed as a real **HTTP API** — including a **streaming** endpoint so responses feel instant. This is the code equivalent of n8n's "Deploy to Production": your agent stops being a notebook and becomes a service other software can call.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/30_deploy_fastapi.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(
    base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"]))
print("Ready:", MODEL_NAME)

## First, streaming — because a blocking agent feels broken

An agent that takes 20 seconds to return a wall of text feels frozen. Streaming shows tokens as they're generated, so the user sees it working. PydanticAI streams with `run_stream` (an async context manager) and `stream_text(delta=True)`, which yields each new chunk. Watch it print live:

In [ ]:
agent = Agent(model, system_prompt="You are a concise assistant.")

async with agent.run_stream("Explain what an API is, in 3 short sentences.") as response:
    async for chunk in response.stream_text(delta=True):   # delta=True -> only the NEW text each step
        print(chunk, end="", flush=True)

That token-by-token flow is what every good chat UI does. Now we wrap the agent in a web API that offers both a plain JSON endpoint and a streaming one.

## The API

We write it to a file with `%%writefile` (a notebook can't host a long-running server cleanly — see the note below). In an `async def` route we use `await agent.run(...)` (never `run_sync`, which errors inside a running event loop). The streaming route uses FastAPI's `StreamingResponse` with Server-Sent Events (`text/event-stream`).

In [ ]:
%%writefile app.py
import os
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

model = OpenAIChatModel(
    "meta-llama/llama-3.3-70b-instruct",
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
agent = Agent(model, system_prompt="You are a concise assistant.")

app = FastAPI()

class Message(BaseModel):
    text: str

@app.post("/chat")               # plain JSON: wait for the full reply
async def chat(msg: Message):
    result = await agent.run(msg.text)
    return {"reply": result.output}

@app.post("/chat/stream")        # streaming: tokens as Server-Sent Events
async def chat_stream(msg: Message):
    async def gen():
        async with agent.run_stream(msg.text) as response:
            async for chunk in response.stream_text(delta=True):
                yield f"data: {chunk}\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

## Run it

Locally (a normal terminal), install and launch:

```bash
pip install fastapi uvicorn "pydantic-ai-slim[openai]>=2.0,<3.0"
export OPENROUTER_API_KEY=sk-or-...
uvicorn app:app --reload
```

Then call it:

```bash
curl -X POST localhost:8000/chat -H 'content-type: application/json' -d '{"text": "Hello!"}'
curl -N -X POST localhost:8000/chat/stream -H 'content-type: application/json' -d '{"text": "Tell me a joke"}'
```

```{note}
**Running a server inside Colab** is awkward — the notebook already owns the event loop, so `uvicorn.run(...)` blocks, and Colab ports aren't public. For real deployment, run the `.py` on a host (Railway, Render, Fly, a VM) exactly as above. To demo from Colab, use `nest_asyncio` + a background thread + a tunnel (ngrok/cloudflared) — but the script-on-a-host path is what you'll actually ship.
```

::::{dropdown} 🔍 The LangGraph version
:color: info

A LangGraph agent deploys the same way — `await graph.ainvoke(...)` for JSON, `graph.astream(..., stream_mode="messages")` for streaming:

```python
@app.post("/chat/stream")
async def stream(msg: Message):
    cfg = {"configurable": {"thread_id": "web-1"}}
    async def gen():
        async for chunk, meta in graph.astream(
                {"messages": [{"role": "user", "content": msg.text}]}, cfg, stream_mode="messages"):
            if chunk.content:
                yield f"data: {chunk.content}\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")
```

LangGraph also has a managed deployment product (LangGraph Platform) if you don't want to run the server yourself.
::::

**What's next:** in **31** we put a real front end in front of this API — the same move as n8n's "Connect Your App."